# 01 — Split Data

**Goal:** Load both raw CSVs, merge them into one clean dataset, and create a chronological train/val/test split.

> ⚠️ This is the only notebook that reads from `data/raw/`.  
> Everything else reads from `data/processed/`.  
> The test set is saved here and will NOT be used again until notebook 07.

**Steps:**
1. Load energy + weather data
2. Aggregate weather (5 cities → 1 row per hour)
3. Merge datasets on timestamp
4. Drop useless columns (100% null, 99.9% zero)
5. Handle missing values in remaining columns
6. Chronological split — 70% train / 15% val / 15% test
7. Validate splits have no overlap
8. Save to `data/processed/`

In [1]:
import pandas as pd
import numpy as np
import os
import json

# ── Paths ─────────────────────────────────────────────────
RAW_PATH       = '../data/raw/'
PROCESSED_PATH = '../data/processed/'
os.makedirs(PROCESSED_PATH, exist_ok=True)

# ── Split ratios ───────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# ── Columns to drop immediately ────────────────────────────
# 100% null → useless
# 99.9% zero → no signal, just noise
DROP_COLS = [
    'generation hydro pumped storage aggregated',  # 100% null
    'forecast wind offshore eday ahead',           # 100% null
    'generation fossil coal-derived gas',          # 99.9% zero
    'generation fossil oil shale',                 # 99.9% zero
    'generation fossil peat',                      # 99.9% zero
    'generation geothermal',                       # 99.9% zero
    'generation marine',                           # 99.9% zero
    'generation wind offshore',                    # 99.9% zero
]

print('Configuration loaded.')

Configuration loaded.


## 1. Load Raw Data

In [2]:
# ── Energy dataset ─────────────────────────────────────────
energy = pd.read_csv(RAW_PATH + 'energy_dataset.csv')

# Parse timestamp — data comes with timezone info, standardize to UTC
energy['time'] = pd.to_datetime(energy['time'], utc=True)

print('Energy dataset:')
print(f'  Shape:    {energy.shape}')
print(f'  From:     {energy["time"].min()}')
print(f'  To:       {energy["time"].max()}')
print(f'  Sorted:   {energy["time"].is_monotonic_increasing}')
energy.head(3)

Energy dataset:
  Shape:    (35064, 29)
  From:     2014-12-31 23:00:00+00:00
  To:       2018-12-31 22:00:00+00:00
  Sorted:   True


,time,generation biomass,generation fossil brown coal/lignite,generation fossil coal-derived gas,generation fossil gas,generation fossil hard coal,generation fossil oil,generation fossil oil shale,generation fossil peat,generation geothermal,...,generation waste,generation wind offshore,generation wind onshore,forecast solar day ahead,forecast wind offshore eday ahead,forecast wind onshore day ahead,total load forecast,total load actual,price day ahead,price actual
0,2014-12-31 23:00:00+00:00,447.0,329.0,0.0,4844.0,4821.0,162.0,0.0,0.0,0.0,...,196.0,0.0,6378.0,17.0,NaN,6436.0,26118.0,25385.0,50.10,65.41
1,2015-01-01 00:00:00+00:00,449.0,328.0,0.0,5196.0,4755.0,158.0,0.0,0.0,0.0,...,195.0,0.0,5890.0,16.0,NaN,5856.0,24934.0,24382.0,48.10,64.92
2,2015-01-01 01:00:00+00:00,448.0,323.0,0.0,4857.0,4581.0,157.0,0.0,0.0,0.0,...,196.0,0.0,5461.0,8.0,NaN,5454.0,23515.0,22734.0,47.33,64.48


In [3]:
# ── Weather dataset ────────────────────────────────────────
weather = pd.read_csv(RAW_PATH + 'weather_features.csv')
weather['dt_iso'] = pd.to_datetime(weather['dt_iso'], utc=True)

print('Weather dataset:')
print(f'  Shape:          {weather.shape}')
print(f'  Unique timestamps: {weather["dt_iso"].nunique()}')
print(f'  Cities:         {weather["city_name"].unique().tolist()}')
print()
print('Rows per city:')
print(weather['city_name'].value_counts().to_string())
weather.head(3)

Weather dataset:
  Shape:          (178396, 17)
  Unique timestamps: 35064
  Cities:         ['Valencia', 'Madrid', 'Bilbao', ' Barcelona', 'Seville']

Rows per city:
city_name
Madrid        36267
Bilbao        35951
Seville       35557
 Barcelona    35476
Valencia      35145


,dt_iso,city_name,temp,temp_min,temp_max,pressure,humidity,wind_speed,wind_deg,rain_1h,rain_3h,snow_3h,clouds_all,weather_id,weather_main,weather_description,weather_icon
0,2014-12-31 23:00:00+00:00,Valencia,270.475,270.475,270.475,1001,77,1,62,0.0,0.0,0.0,0,800,clear,sky is clear,01n
1,2015-01-01 00:00:00+00:00,Valencia,270.475,270.475,270.475,1001,77,1,62,0.0,0.0,0.0,0,800,clear,sky is clear,01n
2,2015-01-01 01:00:00+00:00,Valencia,269.686,269.686,269.686,1002,78,0,23,0.0,0.0,0.0,0,800,clear,sky is clear,01n


## 2. Aggregate Weather — 5 Cities → 1 Row per Hour

Weather has 5 cities × 35,064 hours = 178,396 rows.  
We average numeric features across cities to get 1 row per timestamp.

In [4]:
# Numeric weather features to aggregate
WEATHER_NUMERIC_COLS = [
    'temp', 'temp_min', 'temp_max', 'pressure',
    'humidity', 'wind_speed', 'wind_deg',
    'rain_1h', 'rain_3h', 'snow_3h', 'clouds_all'
]

weather_agg = (
    weather
    .groupby('dt_iso')[WEATHER_NUMERIC_COLS]
    .mean()
    .reset_index()
)

print('Weather after aggregation:')
print(f'  Shape:  {weather_agg.shape}  (was {weather.shape[0]} rows)')
print(f'  Nulls:  {weather_agg.isnull().sum().sum()}')
weather_agg.head(3)

Weather after aggregation:
  Shape:  (35064, 12)  (was 178396 rows)
  Nulls:  0


,dt_iso,temp,temp_min,temp_max,pressure,humidity,wind_speed,wind_deg,rain_1h,rain_3h,snow_3h,clouds_all
0,2014-12-31 23:00:00+00:00,272.491463,272.491463,272.491463,1016.4,82.4,2.0,135.2,0.0,0.0,0.0,0.0
1,2015-01-01 00:00:00+00:00,272.512700,272.512700,272.512700,1016.2,82.4,2.0,135.8,0.0,0.0,0.0,0.0
2,2015-01-01 01:00:00+00:00,272.099137,272.099137,272.099137,1016.8,82.0,2.4,119.0,0.0,0.0,0.0,0.0


## 3. Merge Energy + Weather

In [5]:
merged = energy.merge(
    weather_agg,
    left_on='time',
    right_on='dt_iso',
    how='inner'   # only keep timestamps present in BOTH datasets
)

# dt_iso is a duplicate of time — drop it
merged = merged.drop(columns=['dt_iso'])

# Sort by time — CRITICAL for time-series
merged = merged.sort_values('time').reset_index(drop=True)

print('After merge:')
print(f'  Shape:     {merged.shape}')
print(f'  From:      {merged["time"].min()}')
print(f'  To:        {merged["time"].max()}')
print(f'  Sorted:    {merged["time"].is_monotonic_increasing}')

After merge:
  Shape:     (35064, 40)
  From:      2014-12-31 23:00:00+00:00
  To:        2018-12-31 22:00:00+00:00
  Sorted:    True


## 4. Drop Useless Columns

In [6]:
print('Columns before drop:', merged.shape[1])
print()
print('Dropping:')
for col in DROP_COLS:
    null_pct = merged[col].isnull().sum() / len(merged) * 100
    zero_pct = (merged[col] == 0).sum() / len(merged) * 100
    print(f'  {col:50s}  null={null_pct:.1f}%  zero={zero_pct:.1f}%')

merged = merged.drop(columns=DROP_COLS)
print()
print('Columns after drop:', merged.shape[1])

Columns before drop: 40

Dropping:
  generation hydro pumped storage aggregated          null=100.0%  zero=0.0%
  forecast wind offshore eday ahead                   null=100.0%  zero=0.0%
  generation fossil coal-derived gas                  null=0.1%  zero=99.9%
  generation fossil oil shale                         null=0.1%  zero=99.9%
  generation fossil peat                              null=0.1%  zero=99.9%
  generation geothermal                               null=0.1%  zero=99.9%
  generation marine                                   null=0.1%  zero=99.9%
  generation wind offshore                            null=0.1%  zero=99.9%

Columns after drop: 32


## 5. Handle Missing Values

**Strategy:**
- `total load actual` (target): time-based interpolation — gaps are small (max ~36 rows consecutive)
- Generation columns: forward-fill — small gaps, values are stable hour-to-hour
- Weather columns: no nulls after aggregation

In [7]:
print('Nulls before cleaning:')
nulls = merged.isnull().sum()
print(nulls[nulls > 0].to_string())

Nulls before cleaning:
generation biomass                             19
generation fossil brown coal/lignite           18
generation fossil gas                          18
generation fossil hard coal                    18
generation fossil oil                          19
generation hydro pumped storage consumption    19
generation hydro run-of-river and poundage     19
generation hydro water reservoir               18
generation nuclear                             17
generation other                               18
generation other renewable                     18
generation solar                               18
generation waste                               19
generation wind onshore                        18
total load actual                              36


In [8]:
# Set time as index for time-based interpolation
merged = merged.set_index('time')

# Target: time-weighted interpolation (preserves temporal trend)
merged['total load actual'] = merged['total load actual'].interpolate(method='time')

# Forecasts and generation: forward fill (assume last known value holds)
cols_to_ffill = [
    c for c in merged.columns
    if c != 'total load actual' and merged[c].isnull().sum() > 0
]
merged[cols_to_ffill] = merged[cols_to_ffill].ffill()

# Reset index — time back as column
merged = merged.reset_index()

print('Nulls after cleaning:')
remaining_nulls = merged.isnull().sum()
print(remaining_nulls[remaining_nulls > 0].to_string() if remaining_nulls.sum() > 0 else '  None ✅')
print()
print('Final merged dataset shape:', merged.shape)

Nulls after cleaning:
  None ✅

Final merged dataset shape: (35064, 32)


## 6. Chronological Split

> **Why NOT `train_test_split`?**  
> Random split leaks future data into training — the model learns from timestamps that come AFTER the test period.  
> In production you never have future data when predicting. Chronological split mirrors reality.

In [9]:
n = len(merged)
train_end = int(n * TRAIN_RATIO)
val_end   = int(n * (TRAIN_RATIO + VAL_RATIO))

train = merged.iloc[:train_end].copy()
val   = merged.iloc[train_end:val_end].copy()
test  = merged.iloc[val_end:].copy()

print('Split summary:')
print(f'  Train : {len(train):>6,} rows  |  {train["time"].min().date()} → {train["time"].max().date()}')
print(f'  Val   : {len(val):>6,} rows  |  {val["time"].min().date()} → {val["time"].max().date()}')
print(f'  Test  : {len(test):>6,} rows  |  {test["time"].min().date()} → {test["time"].max().date()}')
print()
print(f'  Total : {len(train) + len(val) + len(test):>6,} rows  (original: {n})')

Split summary:
  Train : 24,544 rows  |  2014-12-31 → 2017-10-19
  Val   :  5,260 rows  |  2017-10-19 → 2018-05-26
  Test  :  5,260 rows  |  2018-05-26 → 2018-12-31

  Total : 35,064 rows  (original: 35064)


## 7. Validate Splits — No Overlap, No Leakage

In [10]:
# These assertions will raise an error if anything is wrong
assert train['time'].max() < val['time'].min(), \
    '❌ Train and Val overlap!'

assert val['time'].max() < test['time'].min(), \
    '❌ Val and Test overlap!'

assert len(train) + len(val) + len(test) == n, \
    '❌ Rows lost during split!'

assert train['time'].is_monotonic_increasing, '❌ Train not sorted!'
assert val['time'].is_monotonic_increasing,   '❌ Val not sorted!'
assert test['time'].is_monotonic_increasing,  '❌ Test not sorted!'

print('✅ All validation checks passed:')
print(f'   Train ends:  {train["time"].max()}')
print(f'   Val starts:  {val["time"].min()}')
print(f'   Val ends:    {val["time"].max()}')
print(f'   Test starts: {test["time"].min()}')
print()
print('🔒 Test set is now LOCKED until notebook 07.')

✅ All validation checks passed:
   Train ends:  2017-10-19 14:00:00+00:00
   Val starts:  2017-10-19 15:00:00+00:00
   Val ends:    2018-05-26 18:00:00+00:00
   Test starts: 2018-05-26 19:00:00+00:00

🔒 Test set is now LOCKED until notebook 07.


## 8. Save to Disk

In [11]:
train.to_csv(PROCESSED_PATH + 'train.csv', index=False)
val.to_csv(PROCESSED_PATH   + 'val.csv',   index=False)
test.to_csv(PROCESSED_PATH  + 'test.csv',  index=False)

# Save metadata for downstream notebooks
meta = {
    'target_col'    : 'total load actual',
    'time_col'      : 'time',
    'train_rows'    : len(train),
    'val_rows'      : len(val),
    'test_rows'     : len(test),
    'train_start'   : str(train['time'].min()),
    'train_end'     : str(train['time'].max()),
    'val_start'     : str(val['time'].min()),
    'val_end'       : str(val['time'].max()),
    'test_start'    : str(test['time'].min()),
    'test_end'      : str(test['time'].max()),
    'all_columns'   : merged.columns.tolist(),
    'dropped_cols'  : DROP_COLS,
    'weather_cols'  : WEATHER_NUMERIC_COLS,
}

with open(PROCESSED_PATH + 'meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('✅ Files saved:')
print(f'   {PROCESSED_PATH}train.csv  →  {len(train):,} rows')
print(f'   {PROCESSED_PATH}val.csv    →  {len(val):,} rows')
print(f'   {PROCESSED_PATH}test.csv   →  {len(test):,} rows')
print(f'   {PROCESSED_PATH}meta.json')
print()
print('→ Proceed to notebook 02_eda.ipynb')

✅ Files saved:
   ../data/processed/train.csv  →  24,544 rows
   ../data/processed/val.csv    →  5,260 rows
   ../data/processed/test.csv   →  5,260 rows
   ../data/processed/meta.json

→ Proceed to notebook 02_eda.ipynb
